# Preparing data

Notebook purpose:
- Impute missing data
- Create additional variables
- Define eligibility and create firm-level covariates for matching
- Match control firms with PSM
- Save dataset for analysis

#### 1. Set-up

In [9]:
import pandas as pd
import numpy as np
import networkx as nx
from functools import reduce

Load paths and data

In [2]:
# paths
from ipcei_spillovers.io import data_path, results_path, load_df, save_df

# logging - needed?
from ipcei_spillovers.config_logging import setup_logging
log = setup_logging(filename="preparation.log")

In [31]:
# ----------------------------
# Load data
# ----------------------------

firm_master_ipcei_events = load_df(
    "interim", "combined", "firm_master_ipcei_events_jan26.csv", low_memory=False
)
firm_master_ipcei = load_df(
    "interim", "combined", "firm_master_ipcei_jan26.csv", low_memory=False
)
partnership_master = load_df(
    "interim", "combined", "partnership_master_ipcei_jan26.csv", low_memory=False
)
firm_month_outcomes = load_df(
    "interim", "combined", "firm_month_outcomes_jan26.csv", low_memory=False
)

#### 2. Prepare data

a) Impute missing data

In [20]:
# ========================================
# Sector, staff count and founding year
# ========================================

# ----------------------------------------
# Define corrections
# ----------------------------------------

# 1 missing sector
# 6 missing staffCount
# 44 missing foundingYear

manual_corrections = [
    {"Linkedin_name": "1s1-energy", "firm_type": "Manufacturer", "sector": "Hydrogen", "industry": "Renewable Energy Semiconductor Manufacturing", "staffCount": 10, "foundingYear": 2019},
    {"Linkedin_name": "advent-technologies-inc", "firm_type": "Manufacturer", "sector": "Hydrogen, Automotive, Other Transport", "industry": "Appliances, Electrical, and Electronics Manufacturing", "staffCount": 42, "foundingYear": 2006},
    {"Linkedin_name": "alstom", "firm_type": "Manufacturer, Service Provider", "sector": "Railway, Other Transport", "industry": "Motor Vehicle Manufacturing", "staffCount": 82359, "foundingYear": 1928},
    {"Linkedin_name": "avl", "firm_type": "Manufacturer, Service Provider", "sector": "Automotive, Electric Vehicles, Battery, Hydrogen, Software, Other ICT", "industry": "Motor Vehicle Manufacturing", "staffCount": 8532, "foundingYear": 1948},
    {"Linkedin_name": "basf", "firm_type": "Manufacturer", "sector": "Chemical Industry, Agriculture, Gas, Oil", "industry": "Chemical Manufacturing", "staffCount": 67892, "foundingYear": 1865},
    {"Linkedin_name": "bmw-group", "firm_type": "Manufacturer, Service Provider", "sector": "Automotive, Other ICT, Software", "industry": "Motor Vehicle Manufacturing", "staffCount": 63276, "foundingYear": 1916},
    {"Linkedin_name": "bosch", "firm_type": "Manufacturer, Service Provider", "sector": "Automotive, Software, Built Environment, Other Sectors", "industry": "Software Development", "staffCount": 162452, "foundingYear": 1886},
    {"Linkedin_name": "bp", "firm_type": "Manufacturer, Utility", "sector": "Oil, Gas", "industry": "Oil and Gas", "staffCount": 100500, "foundingYear": 1909},
    {"Linkedin_name": "btcomposites", "firm_type": "Manufacturer", "sector": "Wind, Hydrogen", "industry": "Machinery Manufacturing", "staffCount": 16, "foundingYear": 2009},
    {"Linkedin_name": "christofindustries", "firm_type": "Manufacturer, Service Provider", "sector": "Oil, Chemical Industry, Agriculture, Automotive, Other Construction, Mining & Metallurgy, Other Sectors", "industry": "Industrial Machinery Manufacturing", "staffCount": 329, "foundingYear": 1966},
    {"Linkedin_name": "creos-deutschland", "firm_type": "Utility", "sector": "Electricity-grid, Gas", "industry": "Utilities", "staffCount": 54, "foundingYear": 2009},
    {"Linkedin_name": "cummins-inc", "firm_type": "Manufacturer", "sector": "Other Manufacturing, Other Energy Storage", "industry": "Motor Vehicle Manufacturing", "staffCount": 51319, "foundingYear": 1919},
    {"Linkedin_name": "daimlertruckag", "firm_type": "Manufacturer", "sector": "Automotive, Other Transport", "industry": "Motor Vehicle Manufacturing", "staffCount": 8890, "foundingYear": 2021},
    {"Linkedin_name": "edpes", "firm_type": "Utility", "sector": "Wind, Solar, Hydro energy, Electricity-grid, Energy General", "industry": "Utilities", "staffCount": 796, "foundingYear": 1976},
    {"Linkedin_name": "eiffageenergiesystemes", "firm_type": "Service Provider", "sector": "Energy General", "industry": "Environmental Services", "staffCount": 14704, "foundingYear": 2017},
    {"Linkedin_name": "en-it-group", "firm_type": "Project Developer", "sector": "Wind, Solar", "industry": "Renewable Energy Semiconductor Manufacturing", "staffCount": 15, "foundingYear": 1993},
    {"Linkedin_name": "enbw", "firm_type": "Utility", "sector": "Energy General, Electric Vehicles", "industry": "Utilities", "staffCount": 5569, "foundingYear": 1997},
    {"Linkedin_name": "endurance-overseas-spa", "firm_type": "Manufacturer", "sector": "Automotive, Other Manufacturing", "industry": "Motor Vehicle Manufacturing", "staffCount": 142, "foundingYear": 2019},
    {"Linkedin_name": "engie", "firm_type": "Utility, Service Provider", "sector": "Energy General, Gas, Electricity-grid", "industry": "Renewable Energy Semiconductor Manufacturing", "staffCount": 44738, "foundingYear": 2008},
    {"Linkedin_name": "engie-belgium", "firm_type": "Utility", "sector": "Energy General, Gas, Other Renewables", "industry": "Utilities", "staffCount": 3291, "foundingYear": 1905},
    {"Linkedin_name": "eni", "firm_type": "Manufacturer, Utility", "sector": "Biofuels, Biogas, Solar, Gas, Oil, Other Renewables", "industry": "Oil and Gas", "staffCount": 31633, "foundingYear": 1953},
    {"Linkedin_name": "ewe-gasspeicher-gmbh", "firm_type": "Utility", "sector": "Gas", "industry": "Utilities", "staffCount": 90, "foundingYear": 2012},
    {"Linkedin_name": "fincantieri", "firm_type": "Manufacturer", "sector": "Shipping, Other Manufacturing", "industry": "Shipbuilding", "staffCount": 6586, "foundingYear": 1959},
    {"Linkedin_name": "fpt-industrial", "firm_type": "Manufacturer", "sector": "Automotive, Other Transport, Other Construction, Agriculture, Other Manufacturing, Energy General", "industry": "Industrial Machinery Manufacturing", "staffCount": 2864, "foundingYear": 2005},
    {"Linkedin_name": "gen-hy", "firm_type": "Manufacturer", "sector": "Hydrogen", "industry": "Renewable Energy Semiconductor Manufacturing", "staffCount": 26, "foundingYear": 2019},
    {"Linkedin_name": "hamburgerenergienetze", "firm_type": "Utility", "sector": "Electricity-grid", "industry": "Utilities", "staffCount": 771, "foundingYear": 1889},
    {"Linkedin_name": "hydrogene-de-france", "firm_type": "Manufacturer", "sector": "Hydrogen", "industry": "Appliances, Electrical, and Electronics Manufacturing", "staffCount": 3, "foundingYear": 2012},  # staffCount missing in your data
    {"Linkedin_name": "iberdrola", "firm_type": "Utility", "sector": "Hydrogen, Wind, Solar, Other Energy Storage, Electricity-grid", "industry": "Renewable Energy Semiconductor Manufacturing", "staffCount": 26847, "foundingYear": 1901},
    {"Linkedin_name": "iqony", "firm_type": "Manufacturer, Service Provider, Project Developer", "sector": "Solar, Wind, Geothermal, Hydrogen, Other Energy Storage, Gas, Energy General", "industry": "IT Services and IT Consulting", "staffCount": 550, "foundingYear": 2022},
    {"Linkedin_name": "iveco-group", "firm_type": "Manufacturer", "sector": "Automotive", "industry": "Motor Vehicle Manufacturing", "staffCount": 19778, "foundingYear": 2022},
    {"Linkedin_name": "john-cockerill", "firm_type": "Manufacturer, Service Provider", "sector": "Energy General, Iron and Steel, Solar, Wind, Hydrogen, Nuclear Energy, Other Sectors", "industry": "Industrial Machinery Manufacturing", "staffCount": 6823, "foundingYear": 2002},
    {"Linkedin_name": "katek-group", "firm_type": "Manufacturer", "sector": "Computer and Electronic Products", "industry": "Appliances, Electrical, and Electronics Manufacturing", "staffCount": 545, "foundingYear": 1976},
    {"Linkedin_name": "liofit-gmbh", "firm_type": "Service Provider", "sector": "Battery, Electric Vehicles", "industry": "Furniture and Home Furnishings Manufacturing", "staffCount": 4, "foundingYear": 2013},
    {"Linkedin_name": "nafta-a-s-", "firm_type": "Utility, Service Provider", "sector": "Gas, Oil, Other Energy Storage", "industry": "Truck Transportation", "staffCount": 194, "foundingYear": 1913},
    {"Linkedin_name": "opmobility", "firm_type": "Manufacturer", "sector": "Solar, Battery, Hydrogen, Automotive", "industry": "Motor Vehicle Manufacturing", "staffCount": 6596, "foundingYear": 1946},
    {"Linkedin_name": "pkn-orlen-s.a.", "firm_type": "Utility", "sector": "Oil, Energy General", "industry": "Oil and Gas", "staffCount": 6580, "foundingYear": 1999},
    {"Linkedin_name": "prayon", "firm_type": "Manufacturer", "sector": "Chemical Industry, Battery", "industry": "Chemical Manufacturing", "staffCount": 1029, "foundingYear": 1882},
    {"Linkedin_name": "rina", "firm_type": "Service Provider", "sector": "Marine Energy, Energy General, Other Transport, Other Construction, Software, Other Sectors", "industry": "IT Services and IT Consulting", "staffCount": 7692, "foundingYear": 1861},
    {"Linkedin_name": "rwe-", "firm_type": "Utility", "sector": "Wind, Solar, Biomass, Hydrogen, Battery, Other Fossil Energy, Energy General, Coal, Nuclear Energy", "industry": "Utilities", "staffCount": 14860, "foundingYear": 1898},
    {"Linkedin_name": "sener", "firm_type": "Service Provider", "sector": "Other Transport, Other Renewables, Shipping", "industry": "IT Services and IT Consulting", "staffCount": 3815, "foundingYear": 1956},
    {"Linkedin_name": "shell", "firm_type": "Utility, Manufacturer", "sector": "Oil, Gas, Chemical Industry", "industry": "Oil and Gas", "staffCount": 166670, "foundingYear": 1907},
    {"Linkedin_name": "sibanye-stillwaters-keliber-lithium-project", "firm_type": "Mining", "sector": "Battery, Other Manufacturing, Raw Materials", "industry": "Mining", "staffCount": 131, "foundingYear": 2023},
    {"Linkedin_name": "stellantis", "firm_type": "Manufacturer", "sector": "Automotive", "industry": "Motor Vehicle Manufacturing", "staffCount": 112770, "foundingYear": 2021},
    {"Linkedin_name": "totalenergies", "firm_type": "Manufacturer, Utility", "sector": "Oil, Gas, Solar, Wind, Other Energy Storage, Shipping, Other Sectors", "industry": "Oil and Gas", "staffCount": 80116, "foundingYear": 1924},
    {"Linkedin_name": "ufihydrogen", "firm_type": "Manufacturer, Research Institute", "sector": "Hydrogen", "industry": "Industrial Machinery Manufacturing", "staffCount": 26, "foundingYear": 2023},
    {"Linkedin_name": "varta-micro-innovation-gmbh", "firm_type": "Manufacturer", "sector": "Battery", "industry": "Chemical Manufacturing", "staffCount": 8, "foundingYear": 2009},
    {"Linkedin_name": "verbund-hydrogen", "firm_type": "Utility, Project Developer", "sector": "Hydrogen", "industry": "Utilities", "staffCount": 22, "foundingYear": 2023},
    {"Linkedin_name": "vopak", "firm_type": "Service Provider", "sector": "Hydrogen, Other Energy Storage, Carbon Capture And Storage", "industry": "Truck Transportation", "staffCount": 4110, "foundingYear": 1616},
    {"Linkedin_name": "zts-v%C3%BDskum-a-v%C3%BDvoj-a-s", "firm_type": "Research Institute, Manufacturer", "sector": "Hydro energy, Battery", "industry": "Industrial Machinery Manufacturing", "staffCount": 7, "foundingYear": 1997},
]

for col in ['firm_type', 'sector', 'industry', 'staffCount', 'foundingYear']:
    correction_flag = f"{col}_manually_corrected"
    if correction_flag not in firm_master_ipcei.columns:
        firm_master_ipcei[correction_flag] = 0

# ----------------------------------------
# Apply corrections
# ----------------------------------------
print("\n" + "="*80)
print("APPLYING MANUAL CORRECTIONS FOR FIRM CHARACTERISTICS")
print("="*80)

n_found = 0
n_not_found = 0
corrections_applied = {
    'firm_type': 0,
    'sector': 0,
    'industry': 0,
    'staffCount': 0,
    'foundingYear': 0
}

for correction in manual_corrections:
    firm_name = correction["Linkedin_name"]

    # Find firm in dataframe
    mask = firm_master_ipcei["Linkedin_name"] == firm_name

    if mask.sum() == 0:
        print(f"⚠️  Firm not found: {firm_name}")
        n_not_found += 1
        continue

    if mask.sum() > 1:
        print(f"⚠️  Multiple matches for: {firm_name} (using first)")

    n_found += 1

    # Track what changed for this firm
    changes = []

    # Apply each correction
    for col in ['firm_type', 'sector', 'industry', 'staffCount', 'foundingYear']:
        if col in correction and correction[col] is not None:
            new_value = correction[col]
            old_value = firm_master_ipcei.loc[mask, col].iloc[0]

            # Only update if different (or if old was missing)
            if pd.isna(old_value) or str(old_value).strip() == "" or old_value != new_value:
                firm_master_ipcei.loc[mask, col] = new_value
                firm_master_ipcei.loc[mask, f"{col}_manually_corrected"] = 1
                corrections_applied[col] += 1

                # Format for display
                if pd.isna(old_value) or str(old_value).strip() == "":
                    old_display = "[MISSING]"
                else:
                    old_display = str(old_value)

                changes.append(f"{col}: '{old_display}' → '{new_value}'")

    # Print summary for this firm
    if changes:
        print(f"\n✓ {firm_name}:")
        for change in changes:
            print(f"  {change}")

# ----------------------------------------
# Summary
# ----------------------------------------
print(f"\n{'='*80}")
print(f"CORRECTION SUMMARY")
print(f"{'='*80}")
print(f"Total corrections attempted: {len(manual_corrections)}")
print(f"Firms found and corrected: {n_found}")
print(f"Firms not found: {n_not_found}")
print(f"\nCorrections by field:")
for col, count in corrections_applied.items():
    print(f"  {col}: {count} firms")

if n_not_found > 0:
    print(f"\n⚠️  Firms not found in dataset - check Linkedin_name spelling:")
    for correction in manual_corrections:
        firm_name = correction["Linkedin_name"]
        if not (firm_master_ipcei["Linkedin_name"] == firm_name).any():
            print(f"  - {firm_name}")

# check for remaining missings
print(f"\n{'='*80}")
print(f"REMAINING MISSING DATA FOR IPCEI FIRMS")
print(f"{'='*80}")

ipcei_firms = firm_master_ipcei[firm_master_ipcei["treated_any"] == 1]

for col in ['firm_type', 'sector', 'industry', 'staffCount', 'foundingYear']:
    missing = ipcei_firms[col].isna() | (ipcei_firms[col] == "")
    n_missing = missing.sum()
    pct_missing = n_missing / len(ipcei_firms) * 100

    print(f"\n{col}:")
    print(f"  Missing: {n_missing} / {len(ipcei_firms)} ({pct_missing:.1f}%)")

    if n_missing > 0 and n_missing <= 10:
        print(f"  Firms still missing {col}:")
        for name in ipcei_firms[missing]["Linkedin_name"].tolist():
            print(f"    - {name}")

# ----------------------------------------
# Save summary
# ----------------------------------------
corrected_firms = firm_master_ipcei[
    (firm_master_ipcei["firm_type_manually_corrected"] == 1) |
    (firm_master_ipcei["sector_manually_corrected"] == 1) |
    (firm_master_ipcei["industry_manually_corrected"] == 1) |
    (firm_master_ipcei["staffCount_manually_corrected"] == 1) |
    (firm_master_ipcei["foundingYear_manually_corrected"] == 1)
]

correction_summary = corrected_firms[[
    "Linkedin_name",
    "firm_type", "firm_type_manually_corrected",
    "sector", "sector_manually_corrected",
    "industry", "industry_manually_corrected",
    "staffCount", "staffCount_manually_corrected",
    "foundingYear", "foundingYear_manually_corrected",
]].copy()


APPLYING MANUAL CORRECTIONS FOR FIRM CHARACTERISTICS

✓ 1s1-energy:
  foundingYear: '[MISSING]' → '2019'

✓ advent-technologies-inc:
  sector: '[MISSING]' → 'Hydrogen, Automotive, Other Transport'

✓ alstom:
  foundingYear: '[MISSING]' → '1928'

✓ avl:
  foundingYear: '[MISSING]' → '1948'

✓ basf:
  foundingYear: '[MISSING]' → '1865'

✓ bmw-group:
  foundingYear: '[MISSING]' → '1916'

✓ bosch:
  foundingYear: '[MISSING]' → '1886'

✓ bp:
  staffCount: '[MISSING]' → '100500'

✓ btcomposites:
  staffCount: '[MISSING]' → '16'

✓ christofindustries:
  foundingYear: '[MISSING]' → '1966'

✓ creos-deutschland:
  foundingYear: '[MISSING]' → '2009'

✓ cummins-inc:
  foundingYear: '[MISSING]' → '1919'

✓ daimlertruckag:
  foundingYear: '[MISSING]' → '2021'

✓ edpes:
  foundingYear: '[MISSING]' → '1976'

✓ eiffageenergiesystemes:
  foundingYear: '[MISSING]' → '2017'

✓ en-it-group:
  foundingYear: '[MISSING]' → '1993'

✓ enbw:
  foundingYear: '[MISSING]' → '1997'

✓ endurance-overseas-spa:
  staf

In [21]:
# ========================================
# Country
# ========================================

print(f"Original firm_master shape: {firm_master_ipcei.shape}")
print(f"IPCEI firms (treated_any=1): {firm_master_ipcei['treated_any'].sum()}")

# ----------------------------------------
# Corrections based on LinkedIn and online research
# ----------------------------------------

manual_corrections = [
    {"Linkedin_name": "1s1-energy", "manual_country_additions": "PT, BR, US", "primary_country_new": "US"},
    {"Linkedin_name": "ae-elemental", "manual_country_additions": "PL, LU", "primary_country_new": "PL"},
    {"Linkedin_name": "air-products-europe", "manual_country_additions": "DZ, AR, BH, BE, BR, CA, CL, CN, CO, CZ, EC, FR, DE, HK, HU, IN, ID, IE, IL, IT, JP, KR, MY, MX, NL, NO, OM, PE, PL, PT, QA, RO, SA, SG, SK, ZA, ES, TH, TT, TR, TW, GB, US, AE, UA, UZ", "primary_country_new": "GB"},
    {"Linkedin_name": "airliquide", "manual_country_additions": "FR, JP, US, US, IN, DE, CN, AE, BR, CA", "primary_country_new": "FR"},
    {"Linkedin_name": "bp", "manual_country_additions": "AZ, BE, CY, DK, DE, EE, ES, FI, FR, DE, GE, GR, HU, IT, LV, LT, LU, NL, NO, PL, PT, RO, SK, RS, CH, GB, AT, CZ, BG, BR, CA, MX, TT, US, EG, IQ, KW, MR, OM, SA, SN, ZA, AE, AU, IN, ID, MY, NZ, PH, SG, KR, TH, VN, CN, JP", "primary_country_new": "GB"},
    {"Linkedin_name": "cummins-inc", "manual_country_additions": "BE, NL, DE, SE, NO, CZ, FR, IT, PL, ES, RO, TR, RS, GB, IE", "primary_country_new": "US"},
    {"Linkedin_name": "enerlis", "manual_country_additions": "PL, ES, LU", "primary_country_new": "ES"},
    {"Linkedin_name": "ferroglobe-plc", "manual_country_additions": "AR, CA, CN, FR, NO, ZA, ES, US, VE, GB", "primary_country_new": "GB"},
    {"Linkedin_name": "hydrogene-de-france", "manual_country_additions": "FR", "primary_country_new": "FR"},
    {"Linkedin_name": "nextchem", "manual_country_additions": "DE, IT, NL", "primary_country_new": "IT"},
    {"Linkedin_name": "opmobility", "manual_country_additions": "AR, AT, BE, BR, CA, CN, CZ, FR, DE, HU, IN, ID, JP, MY, MX, MA, PL, RO, RU, SK, ES, ZA, KR, CH, TH, TR, GB, US", "primary_country_new": "FR"},
    {"Linkedin_name": "titan-cement-group", "manual_country_additions": "GR, EG, US, BR, FR, IT, GB, TR, BG, RS, XK, MK, AL, IN", "primary_country_new": "GR"},
    {"Linkedin_name": "ufihydrogen", "manual_country_additions": "IT", "primary_country_new": "IT"},
    {"Linkedin_name": "uniper-se", "manual_country_additions": "DE, GB, SE, NL", "primary_country_new": "DE"},
       {"Linkedin_name": "btcomposites", "manual_country_additions": "GR", "primary_country_new": "GR"},
    {"Linkedin_name": "endurance-overseas-spa", "manual_country_additions": "IT, DE", "primary_country_new": "IT"},
    {"Linkedin_name": "hamburgerenergienetze", "manual_country_additions": "DE", "primary_country_new": "DE"},
    {"Linkedin_name": "liofit-gmbh", "manual_country_additions": "DE", "primary_country_new": "DE"},
]

# ----------------------------------------
# Helpers
# ----------------------------------------

def parse_country_string(s):
    """Convert country string to set of ISO2 codes"""
    if pd.isna(s) or s == "" or str(s).strip() == "":
        return set()
    # Clean and split
    countries = str(s).split(",")
    cleaned = set()
    for c in countries:
        c_clean = c.strip().upper()
        # Filter out invalid codes
        if c_clean and c_clean not in ["NAN", "OO", ""]:
            cleaned.add(c_clean)
    return cleaned

def country_set_to_string(country_set):
    """Convert set back to comma-separated string"""
    if not country_set:
        return ""
    return ", ".join(sorted(country_set))

# ----------------------------------------
# Application of manual corrections
# ----------------------------------------

print("\n" + "="*80)
print("APPLYING MANUAL COUNTRY CORRECTIONS")
print("="*80)

# Create flag column for tracking
if "country_manually_corrected" not in firm_master_ipcei.columns:
    firm_master_ipcei["country_manually_corrected"] = 0

n_corrections = 0
n_found = 0
n_not_found = 0

for correction in manual_corrections:
    firm_name = correction["Linkedin_name"]
    new_countries_str = correction["manual_country_additions"]
    new_primary = correction["primary_country_new"]

    # Find firm in dataframe
    mask = firm_master_ipcei["Linkedin_name"] == firm_name

    if mask.sum() == 0:
        print(f"⚠️  Firm not found: {firm_name}")
        n_not_found += 1
        continue

    if mask.sum() > 1:
        print(f"⚠️  Multiple matches for: {firm_name} (using first)")

    n_found += 1

    # Get existing countries
    existing = parse_country_string(firm_master_ipcei.loc[mask, "countries"].iloc[0])

    # Parse new countries from manual research
    new_countries = parse_country_string(new_countries_str)

    # Combine (union)
    all_countries = existing | new_countries

    # Update countries string
    old_countries_str = firm_master_ipcei.loc[mask, "countries"].iloc[0]
    new_countries_str_final = country_set_to_string(all_countries)

    firm_master_ipcei.loc[mask, "countries"] = new_countries_str_final
    firm_master_ipcei.loc[mask, "n_countries"] = len(all_countries)

    # Update primary_country if provided and different
    old_primary = firm_master_ipcei.loc[mask, "primary_country"].iloc[0]

    if new_primary and str(new_primary).strip():
        new_primary_clean = str(new_primary).strip().upper()
        if new_primary_clean not in ["NAN", "OO", ""]:
            # Only update if different
            if pd.isna(old_primary) or str(old_primary).strip().upper() != new_primary_clean:
                firm_master_ipcei.loc[mask, "primary_country"] = new_primary_clean
                primary_changed = True
            else:
                primary_changed = False
        else:
            primary_changed = False
    else:
        primary_changed = False

    # Mark as manually corrected
    firm_master_ipcei.loc[mask, "country_manually_corrected"] = 1

    # Log the change
    print(f"\n✓ {firm_name}:")
    print(f"  Old countries: '{old_countries_str}' ({len(existing)} countries)")
    print(f"  New countries: '{new_countries_str_final}' ({len(all_countries)} countries)")
    if primary_changed:
        print(f"  Primary country: '{old_primary}' → '{new_primary_clean}'")

    n_corrections += 1

print(f"\n{'='*80}")
print(f"CORRECTION SUMMARY")
print(f"{'='*80}")
print(f"Total corrections attempted: {len(manual_corrections)}")
print(f"Firms found and corrected: {n_found}")
print(f"Firms not found: {n_not_found}")

if n_not_found > 0:
    print("\n⚠️  Firms not found in dataset - check Linkedin_name spelling:")
    for correction in manual_corrections:
        firm_name = correction["Linkedin_name"]
        if not (firm_master_ipcei["Linkedin_name"] == firm_name).any():
            print(f"  - {firm_name}")

# ----------------------------------------
# Summary stats
# ----------------------------------------

print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

print(f"\nTotal firms: {len(firm_master_ipcei)}")
print(f"IPCEI firms (treated_any=1): {firm_master_ipcei['treated_any'].sum()}")

n_manual = firm_master_ipcei["country_manually_corrected"].sum()
print(f"Firms with manual country corrections: {n_manual}")

# Country completeness
missing_countries = (
    firm_master_ipcei["countries"].isna() |
    (firm_master_ipcei["countries"] == "") |
    (firm_master_ipcei["countries"].str.upper().isin(["NAN", "OO"]))
)

n_missing = missing_countries.sum()
pct_missing = n_missing / len(firm_master_ipcei) * 100
print(f"\nFirms with missing 'countries': {n_missing} ({pct_missing:.1f}%)")

# IPCEI firms with missing countries
ipcei_missing = firm_master_ipcei[
    (firm_master_ipcei["treated_any"] == 1) & missing_countries
]

if len(ipcei_missing) > 0:
    print(f"\n⚠️  IPCEI firms still missing countries: {len(ipcei_missing)}")
    print("\nFirms that still need correction:")
    for idx, row in ipcei_missing.iterrows():
        print(f"\n  {row['Linkedin_name']}:")
        print(f"    current countries: '{row['countries']}'")
        print(f"    primary_country: {row['primary_country']}")
        print(f"    hq_country: {row['hq_country']}")
else:
    print(f"\n✓ All IPCEI firms have country information")

# Distribution of n_countries for IPCEI firms
print(f"\nCountry distribution for IPCEI firms:")
ipcei_firms = firm_master_ipcei[firm_master_ipcei["treated_any"] == 1]
country_dist = ipcei_firms["n_countries"].value_counts().sort_index()
for n, count in country_dist.items():
    print(f"  {int(n)} countries: {count} firms")

# Average number of countries
mean_countries = ipcei_firms["n_countries"].mean()
print(f"\nAverage countries per IPCEI firm: {mean_countries:.1f}")

# Manually corrected IPCEI firms
ipcei_manual = ipcei_firms[ipcei_firms["country_manually_corrected"] == 1]
print(f"IPCEI firms with manual corrections: {len(ipcei_manual)} ({len(ipcei_manual)/len(ipcei_firms)*100:.1f}%)")

# ----------------------------------------
# Save summary
# ----------------------------------------

changes_summary = []

for correction in manual_corrections:
    firm_name = correction["Linkedin_name"]
    mask = firm_master_ipcei["Linkedin_name"] == firm_name

    if mask.sum() > 0:
        row = firm_master_ipcei.loc[mask].iloc[0]
        changes_summary.append({
            "Linkedin_name": firm_name,
            "countries_final": row["countries"],
            "n_countries": row["n_countries"],
            "primary_country": row["primary_country"],
            "manually_corrected": row["country_manually_corrected"]
        })

changes_df = pd.DataFrame(changes_summary)

save_df(
    changes_df,
    "interim",
    "ipcei",
    "ipcei_firms_country_summary.csv"
)

print(f"\n✓ Saved correction summary to ipcei_firms_country_summary.csv")

Original firm_master shape: (239782, 36)
IPCEI firms (treated_any=1): 150

APPLYING MANUAL COUNTRY CORRECTIONS

✓ 1s1-energy:
  Old countries: 'nan' (0 countries)
  New countries: 'BR, PT, US' (3 countries)
  Primary country: 'nan' → 'US'

✓ ae-elemental:
  Old countries: 'nan' (0 countries)
  New countries: 'LU, PL' (2 countries)
  Primary country: 'nan' → 'PL'

✓ air-products-europe:
  Old countries: 'GB' (1 countries)
  New countries: 'AE, AR, BE, BH, BR, CA, CL, CN, CO, CZ, DE, DZ, EC, ES, FR, GB, HK, HU, ID, IE, IL, IN, IT, JP, KR, MX, MY, NL, NO, OM, PE, PL, PT, QA, RO, SA, SG, SK, TH, TR, TT, TW, UA, US, UZ, ZA' (46 countries)

✓ airliquide:
  Old countries: 'BR, CA, FR, JP, NL' (5 countries)
  New countries: 'AE, BR, CA, CN, DE, FR, IN, JP, NL, US' (10 countries)

✓ bp:
  Old countries: 'BR, GB, US' (3 countries)
  New countries: 'AE, AT, AU, AZ, BE, BG, BR, CA, CH, CN, CY, CZ, DE, DK, EE, EG, ES, FI, FR, GB, GE, GR, HU, ID, IN, IQ, IT, JP, KR, KW, LT, LU, LV, MR, MX, MY, NL, N

In [ ]:
# ============================================================
# Save as corrected version
# ============================================================

save_df(
    firm_master_ipcei,
    "interim",
    "combined",
    "firm_master_ipcei_jan26_corrected.csv"  # Overwrite
)

print("✓ Saved: firm_master_ipcei_jan26_corrected.csv")

In [ ]:
# Load corrected firm master
firm_master_ipcei = load_df(
    "interim", "combined", "firm_master_ipcei_jan26_corrected.csv", low_memory=False
)

b) Transform variables

In [22]:
# ========================================
# Log staff count
# ========================================
firm_master_ipcei["log_staffCount"] = np.log(
    firm_master_ipcei["staffCount"].fillna(0) + 1
)

# ----------------------------------------
# Create staffCount buckets (EU SME classification)
# ----------------------------------------

def assign_size_bucket(staff_count):
    """
    Assign firm size bucket based on EU SME definition.

    Categories:
    - Micro: 1-9 employees
    - Small: 10-49 employees
    - Medium: 50-249 employees
    - Large: 250+ employees
    """
    if pd.isna(staff_count) or staff_count == 0:
        return "Unknown"
    elif staff_count < 10:
        return "Micro (1-9)"
    elif staff_count < 50:
        return "Small (10-49)"
    elif staff_count < 250:
        return "Medium (50-249)"
    else:
        return "Large (250+)"

firm_master_ipcei["staffCount_bucket"] = firm_master_ipcei["staffCount"].apply(assign_size_bucket)

# Create numeric version for continuous matching
size_bucket_order = {
    "Unknown": 0,
    "Micro (1-9)": 1,
    "Small (10-49)": 2,
    "Medium (50-249)": 3,
    "Large (250+)": 4
}
firm_master_ipcei["staffCount_bucket_num"] = firm_master_ipcei["staffCount_bucket"].map(size_bucket_order)

print("\nFirm size distribution:")
print(firm_master_ipcei["staffCount_bucket"].value_counts().sort_index())

print("\nTreated firms - size distribution:")
print(firm_master_ipcei[firm_master_ipcei["treated_any"]==1]["staffCount_bucket"].value_counts())

In [ ]:
# ========================================
# Firm age
# ========================================
REFERENCE_YEAR = 2024 # later baseline year to capture firms founded after 2020

firm_master_ipcei["age"] = REFERENCE_YEAR - firm_master_ipcei["foundingYear"]
firm_master_ipcei.loc[firm_master_ipcei["age"] < 0, "age"] = np.nan

# Drop firms with foundingYear < 400 (errors)
# Keep historical entities with foundingYear >= 400 (e.g., République Française founded 496)
firm_master_ipcei.loc[firm_master_ipcei["foundingYear"] < 500, "age"] = np.nan

print(f"Firms with cleaned age (set to NaN): {firm_master_ipcei['age'].isna().sum()}")

c) Create new variables

In [6]:
# ============================================================
# GEOGRAPHIC OPERATIONS
# ============================================================

print("\n" + "="*80)
print("COMPUTING GEOGRAPHIC VARIABLES")
print("="*80)

EU_CODES = {
    "AT","BE","BG","HR","CY","CZ","DK","EE","FI","FR","DE","GR",
    "HU","IE","IT","LV","LT","LU","MT","NL","PL","PT","RO","SK",
    "SI","ES","SE"
}

years = list(range(2020, 2026))

# ── EU operation dummy ──────────────────────────────────────────────
def has_eu_op(country_str):
    if pd.isna(country_str):
        return 0
    countries = {c.strip().upper() for c in str(country_str).split(",")}
    return int(len(countries & EU_CODES) > 0)

firm_master_ipcei["has_eu_operation"] = firm_master_ipcei["countries"].apply(has_eu_op)

# ── log n_countries ─────────────────────────────────────────────────
firm_master_ipcei["log_n_countries"] = np.log(firm_master_ipcei["n_countries"] + 1)

# ── Primary country is EU ────────────────────────────────────────────────
def primary_is_eu(primary_str):
    if pd.isna(primary_str) or str(primary_str).strip() in ("", "nan"):
        return 0
    return int(str(primary_str).strip().upper() in EU_CODES)

firm_master_ipcei["primary_eu"] = firm_master_ipcei["primary_country"].apply(primary_is_eu)

# ── EU cross-border collaboration counts (annual) ────────────────────────

p = partnership_master.copy()
p["post_date"] = pd.to_datetime(p["post_date"], errors="coerce")
p = p.dropna(subset=["post_date", "source", "target", "collaboration_id"])
p["year"]  = p["post_date"].dt.year
p["month"] = p["post_date"].dt.to_period("M").dt.to_timestamp()

# Build country maps from firm_master_ipcei
def parse_countries(x):
    if pd.isna(x) or str(x).strip() in ("", "nan"):
        return set()
    return {c.strip().upper() for c in str(x).split(",")}

def parse_primary(x):
    if pd.isna(x) or str(x).strip() in ("", "nan"):
        return None
    return str(x).strip().upper()

country_map = dict(zip(
    firm_master_ipcei["Linkedin_name"],
    firm_master_ipcei["countries"].apply(parse_countries)
))
primary_map = dict(zip(
    firm_master_ipcei["Linkedin_name"],
    firm_master_ipcei["primary_country"].apply(parse_primary)
))

# EU flags per endpoint (any EU operation)
p["src_eu"] = p["source"].map(country_map).apply(
    lambda x: bool(x & EU_CODES) if isinstance(x, set) else False
)
p["tgt_eu"] = p["target"].map(country_map).apply(
    lambda x: bool(x & EU_CODES) if isinstance(x, set) else False
)

# EU flags per endpoint (primary country)
p["src_eu_primary"] = p["source"].map(primary_map).apply(
    lambda x: x in EU_CODES if x else False
)
p["tgt_eu_primary"] = p["target"].map(primary_map).apply(
    lambda x: x in EU_CODES if x else False
)
p["src_primary_val"] = p["source"].map(primary_map)
p["tgt_primary_val"] = p["target"].map(primary_map)

# Classification flags
p["is_eu_ecosystem"]    = p["src_eu"] & p["tgt_eu"]
p["is_eu_primary"]      = p["src_eu_primary"] & p["tgt_eu_primary"]
p["is_eu_crossborder"]  = (
    p["src_eu_primary"] &
    p["tgt_eu_primary"] &
    (p["src_primary_val"] != p["tgt_primary_val"])
)
p["is_non_eu"] = p["src_eu"] ^ p["tgt_eu"]

print("✓ Partnership EU flags computed")
print(f"  EU ecosystem:   {p['is_eu_ecosystem'].sum():,} partnerships")
print(f"  EU primary:     {p['is_eu_primary'].sum():,} partnerships")
print(f"  EU crossborder: {p['is_eu_crossborder'].sum():,} partnerships")
print(f"  Non-EU:         {p['is_non_eu'].sum():,} partnerships")

# Annual EU collab counts per firm
def _stack_annual(df_sub, col_name):
    src = df_sub[["year","collaboration_id","source"]].rename(columns={"source":"Linkedin_name"})
    tgt = df_sub[["year","collaboration_id","target"]].rename(columns={"target":"Linkedin_name"})
    ft  = pd.concat([src, tgt], ignore_index=True)
    ft  = ft[ft["year"].isin(years)]
    counts = (
        ft.groupby(["Linkedin_name","year"])["collaboration_id"]
          .nunique().reset_index(name=col_name)
    )
    wide = counts.pivot(index="Linkedin_name", columns="year", values=col_name).fillna(0)
    wide.columns = [f"{col_name}_{int(c)}" for c in wide.columns]
    return wide.reset_index()

eu_eco_df  = _stack_annual(p[p["is_eu_ecosystem"]],   "eu_ecosystem_collabs")
eu_prim_df = _stack_annual(p[p["is_eu_primary"]],     "eu_primary_collabs")
eu_cb_df   = _stack_annual(p[p["is_eu_crossborder"]], "eu_crossborder_collabs")
non_eu_df  = _stack_annual(p[p["is_non_eu"]],         "non_eu_collabs")

for df_add in [eu_eco_df, eu_prim_df, eu_cb_df, non_eu_df]:
    firm_master_ipcei = firm_master_ipcei.merge(df_add, on="Linkedin_name", how="left")

firm_master_ipcei = firm_master_ipcei.fillna(0)

# Baseline focus variables (2020)
firm_master_ipcei["eu_crossborder_collabs_pre"] = (
    firm_master_ipcei.get("eu_crossborder_collabs_2020", 0)
)

print("✓ EU collaboration variables merged into firm_master_ipcei")


COMPUTING GEOGRAPHIC VARIABLES
✓ Partnership EU flags computed
  EU ecosystem:   1,623,599 partnerships
  EU primary:     1,470,846 partnerships
  EU crossborder: 615,210 partnerships
  Non-EU:         1,117,327 partnerships
✓ EU collaboration variables merged into firm_master_ipcei


In [25]:
# ==========================================
# NACE 1-digit proxy classification
# ==========================================

def classify_nace1_full(industry, firm_type=None, sector=None):
    """
    Map firms to NACE 1-digit proxy using LinkedIn fields.

    Designed for:
    - Descriptive statistics
    - Matching covariates
    - DiD controls
    """

    text = " ".join([
        str(industry),
        str(firm_type),
        str(sector)
    ]).lower()

    # ------------------------------------------------------------
    # A — Agriculture
    # ------------------------------------------------------------
    if any(k in text for k in ["agriculture", "farming"]):
        return "A — Agriculture"

    # ------------------------------------------------------------
    # B — Mining
    # ------------------------------------------------------------
    if any(k in text for k in ["mining", "metals", "raw material"]):
        return "B — Mining"

    # ------------------------------------------------------------
    # C — Manufacturing
    # ------------------------------------------------------------
    if any(k in text for k in [
        "manufacturing", "machinery", "industrial",
        "chemical", "electronics", "electrical",
        "appliances", "materials", "battery"
    ]):
        return "C — Manufacturing"

    # ------------------------------------------------------------
    # D — Energy
    # ------------------------------------------------------------
    if any(k in text for k in [
        "utility", "energy", "power", "oil", "gas",
        "renewable", "electricity"
    ]):
        return "D — Energy"

    # ------------------------------------------------------------
    # E — Water / environmental
    # ------------------------------------------------------------
    if any(k in text for k in ["environmental", "water", "waste"]):
        return "E — Environmental"

    # ------------------------------------------------------------
    # F — Construction
    # ------------------------------------------------------------
    if any(k in text for k in ["construction", "infrastructure"]):
        return "F — Construction"

    # ------------------------------------------------------------
    # G — Trade
    # ------------------------------------------------------------
    if any(k in text for k in ["wholesale", "retail", "trade"]):
        return "G — Trade"

    # ------------------------------------------------------------
    # H — Transport
    # ------------------------------------------------------------
    if any(k in text for k in [
        "transport", "automotive", "motor vehicle",
        "mobility", "shipping", "aviation", "rail"
    ]):
        return "H — Transport"

    # ------------------------------------------------------------
    # J — Information
    # ------------------------------------------------------------
    if any(k in text for k in ["software", "it", "digital"]):
        return "J — Information"

    # ------------------------------------------------------------
    # K — Finance
    # ------------------------------------------------------------
    if any(k in text for k in ["financial", "bank", "investment"]):
        return "K — Finance"

    # ------------------------------------------------------------
    # M — Professional & scientific
    # ------------------------------------------------------------
    if any(k in text for k in [
        "research", "consulting", "engineering",
        "nanotechnology", "scientific"
    ]):
        return "M — Professional"

    # ------------------------------------------------------------
    # O — Public sector
    # ------------------------------------------------------------
    if any(k in text for k in ["government", "public"]):
        return "O — Public"

    return "Other"


print("\nApplying FULL NACE proxy classification...")

firm_master_ipcei["nace1_proxy"] = firm_master_ipcei.apply(
    lambda row: classify_nace1_full(
        row.get("industry_manually_corrected") or row.get("industry"),
        row.get("firm_type_manually_corrected") or row.get("firm_type"),
        row.get("sector_manually_corrected") or row.get("sector")
    ),
    axis=1
)

print("✓ NACE full classification complete")


# ==========================================
# Diagnostics
# ==========================================

print("\nDistribution:")
print(firm_master_ipcei["nace1_proxy"].value_counts())

print("\nTreated distribution:")
print(
    firm_master_ipcei.loc[
        firm_master_ipcei["treated_any"] == 1,
        "nace1_proxy"
    ].value_counts()
)

print("\nOverlap check:")
print(
    pd.crosstab(
        firm_master_ipcei["nace1_proxy"],
        firm_master_ipcei["treated_any"]
    )
)


Applying FULL NACE proxy classification...
✓ NACE full classification complete

Distribution:
nace1_proxy
C — Manufacturing    62862
J — Information      41525
Other                30796
D — Energy           24220
H — Transport        16945
O — Public           15068
F — Construction     12797
K — Finance           7221
A — Agriculture       7155
M — Professional      6799
G — Trade             5296
E — Environmental     5225
B — Mining            3873
Name: count, dtype: int64

Treated distribution:
nace1_proxy
C — Manufacturing    91
D — Energy           42
B — Mining            7
A — Agriculture       3
H — Transport         3
E — Environmental     1
J — Information       1
K — Finance           1
M — Professional      1
Name: count, dtype: int64

Overlap check:
treated_any            0   1
nace1_proxy                 
A — Agriculture     7152   3
B — Mining          3866   7
C — Manufacturing  62771  91
D — Energy         24178  42
E — Environmental   5224   1
F — Construction   1

In [26]:
# ==========================================
# Collapse NACE for matching
# ==========================================

collapse_map = {
    "C — Manufacturing": "Production",
    "B — Mining": "Production",

    "D — Energy": "Energy system",
    "E — Environmental": "Energy system",

    "H — Transport": "Transport",

    "J — Information": "Knowledge economy",
    "K — Finance": "Knowledge economy",
    "M — Professional": "Knowledge economy",

    "A — Agriculture": "Primary sector",

    "F — Construction": "Services / Public",
    "G — Trade": "Services / Public",
    "O — Public": "Services / Public",
    "Other": "Services / Public"
}

firm_master_ipcei["nace1_psm"] = firm_master_ipcei["nace1_proxy"].map(collapse_map)

print("\nPSM sector distribution:")
print(firm_master_ipcei["nace1_psm"].value_counts())

print("\nTreated distribution:")
print(
    firm_master_ipcei.loc[firm_master_ipcei["treated_any"] == 1, "nace1_psm"]
    .value_counts()
)


PSM sector distribution:
nace1_psm
Production           66735
Services / Public    63957
Knowledge economy    55545
Energy system        29445
Transport            16945
Primary sector        7155
Name: count, dtype: int64

Treated distribution:
nace1_psm
Production           98
Energy system        43
Primary sector        3
Transport             3
Knowledge economy     3
Name: count, dtype: int64


In [7]:
# ============================================================
# Clean industry variables (more granular than NACE-1)
# ============================================================

print("\n" + "="*80)
print("COMPUTING INDUSTRY_CLEAN")
print("="*80)

KEEP_INDUSTRIES = {
    "Utilities",
    "Motor Vehicle Manufacturing",
    "Industrial Machinery Manufacturing",
    "Chemical Manufacturing",
    "Oil and Gas",
    "Renewable Energy Semiconductor Manufacturing",
    "Appliances, Electrical, and Electronics Manufacturing",
    "IT Services and IT Consulting",
    "Mining",
    "Machinery Manufacturing",
    "Truck Transportation",
    "Environmental Services",
    "Research Services",
}

firm_master_ipcei["industry_clean"] = firm_master_ipcei["industry"].apply(
    lambda x: x if x in KEEP_INDUSTRIES else None
)

# Coverage checks
partnership_firms = set(p["source"]).union(set(p["target"]))

cov_partners = firm_master_ipcei[
    firm_master_ipcei["Linkedin_name"].isin(partnership_firms)
]["industry_clean"].notna().mean()

cov_treated = firm_master_ipcei[
    firm_master_ipcei["treated_any"].fillna(0).astype(int) == 1
]["industry_clean"].notna().mean()

print(f"Coverage among partnership firms: {cov_partners:.1%}")
print(f"Coverage among treated firms:     {cov_treated:.1%}")
print(f"\nindustry_clean distribution (treated firms):")
print(
    firm_master_ipcei[firm_master_ipcei["treated_any"].fillna(0).astype(int)==1]
    ["industry_clean"].value_counts(dropna=False)
)

print("✓ industry_clean created")


COMPUTING INDUSTRY_CLEAN
Coverage among partnership firms: 42.8%
Coverage among treated firms:     92.7%

industry_clean distribution (treated firms):
industry_clean
Utilities                                                26
Motor Vehicle Manufacturing                              22
Industrial Machinery Manufacturing                       18
Chemical Manufacturing                                   17
Renewable Energy Semiconductor Manufacturing             13
Oil and Gas                                              13
None                                                     11
Appliances, Electrical, and Electronics Manufacturing     9
IT Services and IT Consulting                             5
Mining                                                    5
Machinery Manufacturing                                   4
Truck Transportation                                      3
Environmental Services                                    2
Research Services                                    

In [27]:
# ============================================================
# Yearly collaboration counts (2020–2025)
# ============================================================

# Ensure datetime
partnership_master['post_date'] = pd.to_datetime(
    partnership_master['post_date'],
    errors='coerce'
)

# ----------------------------------------
# Stack collaborations (source + target)
# ----------------------------------------

src = partnership_master[['source', 'collaboration_id', 'post_date']] \
    .rename(columns={'source': 'Linkedin_name'})

tgt = partnership_master[['target', 'collaboration_id', 'post_date']] \
    .rename(columns={'target': 'Linkedin_name'})

collab_long = pd.concat([src, tgt], ignore_index=True)

# Extract year
collab_long['year'] = collab_long['post_date'].dt.year

# Restrict to relevant years
years = list(range(2020, 2026))
collab_long = collab_long[collab_long['year'].isin(years)]

# ----------------------------------------
# Count collaborations per firm-year
# ----------------------------------------

collab_counts = (
    collab_long
    .groupby(['Linkedin_name', 'year'])['collaboration_id']
    .nunique()
    .reset_index(name='collab_count')
)

# Pivot to wide
collab_wide = collab_counts.pivot(
    index='Linkedin_name',
    columns='year',
    values='collab_count'
)

collab_wide = collab_wide.fillna(0).astype(int)

# Rename columns
collab_wide.columns = [f'collab_count_{int(c)}' for c in collab_wide.columns]

collab_wide = collab_wide.reset_index()

print(f"✓ Created yearly counts for {len(collab_wide):,} firms")

# ----------------------------------------
# Merge into firm master
# ----------------------------------------

firm_master_ipcei = firm_master_ipcei.merge(
    collab_wide,
    on='Linkedin_name',
    how='left'
)

# Fill missing firms with zero
for y in years:
    col = f'collab_count_{y}'
    if col not in firm_master_ipcei:
        firm_master_ipcei[col] = 0
    else:
        firm_master_ipcei[col] = firm_master_ipcei[col].fillna(0)

# ----------------------------------------
# Log transforms
# ----------------------------------------

for y in years:
    firm_master_ipcei[f'log_collab_count_{y}'] = np.log(
        firm_master_ipcei[f'collab_count_{y}'] + 1
    )

print("✓ Log variables created")

# ============================================================
# Pre-treatment windows derived from yearly counts
# ============================================================

print("\nDeriving baseline windows...")

# Battery baseline: 2020
firm_master_ipcei['collab_pretreat_battery'] = firm_master_ipcei['collab_count_2020']
firm_master_ipcei['log_collab_pretreat_battery'] = np.log(
    firm_master_ipcei['collab_pretreat_battery'] + 1
)

# Hydrogen baseline: 2021
firm_master_ipcei['collab_pretreat_hydrogen'] = firm_master_ipcei['collab_count_2021']
firm_master_ipcei['log_collab_pretreat_hydrogen'] = np.log(
    firm_master_ipcei['collab_pretreat_hydrogen'] + 1
)

print("✓ Baseline variables created")

# ============================================================
# Diagnostics
# ============================================================

print("\n" + "="*80)
print("COLLABORATION DISTRIBUTION CHECK")
print("="*80)

for y in years:
    col = f'collab_count_{y}'
    print(f"\n{col}:")
    print(f"  Mean: {firm_master_ipcei[col].mean():.2f}")
    print(f"  Median: {firm_master_ipcei[col].median():.0f}")
    print(f"  Zero share: {(firm_master_ipcei[col] == 0).mean():.2%}")

print("\n✓ Collaboration variables ready")

print("\n" + "="*80)
print("TREATED FIRMS — BASELINE COLLABORATION")
print("="*80)

treated = firm_master_ipcei[firm_master_ipcei["treated_any"] == 1]

for y in [2020, 2021]:
    col = f'collab_count_{y}'
    print(f"\n{col}:")
    print(f"  Mean: {treated[col].mean():.2f}")
    print(f"  Median: {treated[col].median():.0f}")
    print(f"  Zero share: {(treated[col] == 0).mean():.2%}")

✓ Created yearly counts for 217,109 firms
✓ Log variables created

Deriving baseline windows...
✓ Baseline variables created

COLLABORATION DISTRIBUTION CHECK

collab_count_2020:
  Mean: 0.30
  Median: 0
  Zero share: 91.53%

collab_count_2021:
  Mean: 0.68
  Median: 0
  Zero share: 85.23%

collab_count_2022:
  Mean: 1.09
  Median: 0
  Zero share: 79.53%

collab_count_2023:
  Mean: 1.76
  Median: 0
  Zero share: 71.06%

collab_count_2024:
  Mean: 1.97
  Median: 0
  Zero share: 74.49%

collab_count_2025:
  Mean: 1.73
  Median: 0
  Zero share: 77.60%

✓ Collaboration variables ready

TREATED FIRMS — BASELINE COLLABORATION

collab_count_2020:
  Mean: 17.82
  Median: 2
  Zero share: 40.00%

collab_count_2021:
  Mean: 44.76
  Median: 13
  Zero share: 22.67%


In [28]:
# add extensive margin

firm_master_ipcei["any_collab_2020"] = (
    firm_master_ipcei["collab_count_2020"] > 0
).astype(int)

firm_master_ipcei["any_collab_2021"] = (
    firm_master_ipcei["collab_count_2021"] > 0
).astype(int)

In [30]:
# ============================================================
# CENTRALITIES — Degree + Betweenness (yearly, deduplicated)
# ============================================================

print("\n" + "="*80)
print("COMPUTING NETWORK CENTRALITIES")
print("="*80)

# Ensure datetime
partnership_master['post_date'] = pd.to_datetime(
    partnership_master['post_date'], errors='coerce'
)

# Extract year
partnership_master['year'] = partnership_master['post_date'].dt.year

years = list(range(2020, 2026))

# Deduplicate by unordered pair within year
pm = partnership_master.copy()
pm['pair'] = pm.apply(
    lambda x: tuple(sorted([x['source'], x['target']])),
    axis=1
)

pm = pm.drop_duplicates(subset=['pair', 'year'])

# Prepare storage
centrality_results = []

for year in years:

    print(f"Processing network for {year}...")

    df_year = pm[pm['year'] == year]

    # Build graph
    G = nx.from_pandas_edgelist(df_year, source='source', target='target')

    # Degree centrality
    degree = nx.degree_centrality(G)

    # Betweenness centrality (approximation for speed)
    betweenness = nx.betweenness_centrality(G, k=min(500, len(G.nodes())), seed=42)

    df_cent = pd.DataFrame({
        'Linkedin_name': list(G.nodes()),
        f'degree_centrality_{year}': [degree.get(n, 0) for n in G.nodes()],
        f'betweenness_centrality_{year}': [betweenness.get(n, 0) for n in G.nodes()]
    })

    centrality_results.append(df_cent)

# Merge all years
from functools import reduce

centralities = reduce(
    lambda left, right: pd.merge(left, right, on='Linkedin_name', how='outer'),
    centrality_results
)

# Merge into firm master
firm_master_ipcei = firm_master_ipcei.merge(
    centralities,
    on='Linkedin_name',
    how='left'
)

# Fill missing with zero
for col in centralities.columns:
    if col != "Linkedin_name":
        firm_master_ipcei[col] = firm_master_ipcei[col].fillna(0)

print("✓ Centralities merged into firm master")


COMPUTING NETWORK CENTRALITIES
Processing network for 2020...
Processing network for 2021...
Processing network for 2022...
Processing network for 2023...
Processing network for 2024...
Processing network for 2025...
✓ Centralities merged into firm master


In [34]:
# ============================================================
# LOG CENTRALITIES
# ============================================================

for y in years:
    deg = f"degree_centrality_{y}"
    bet = f"betweenness_centrality_{y}"

    # safe even when values are 0
    firm_master_ipcei[f"log_{deg}"] = np.log1p(firm_master_ipcei[deg])
    firm_master_ipcei[f"log_{bet}"] = np.log1p(firm_master_ipcei[bet])

print("✓ Log centralities created")

✓ Log centralities created


In [10]:
# ============================================================
# TECH COLLABORATION COUNTS
# ============================================================

print("\n" + "="*80)
print("COMPUTING TECH COLLAB COUNTS")
print("="*80)

# If running standalone, re-run the p = partnership_master.copy() block above

techs = ["Hydrogen", "Battery"]
tech_results = []

for tech in techs:
    df_tech = p[p[tech] == 1].copy()

    src = df_tech[["source","collaboration_id","year"]].rename(columns={"source":"Linkedin_name"})
    tgt = df_tech[["target","collaboration_id","year"]].rename(columns={"target":"Linkedin_name"})
    stacked = pd.concat([src, tgt], ignore_index=True)
    stacked = stacked[stacked["year"].isin(years)]

    counts = (
        stacked.groupby(["Linkedin_name","year"])["collaboration_id"]
               .nunique().reset_index(name=f"{tech.lower()}_collabs")
    )
    wide = counts.pivot(
        index="Linkedin_name", columns="year",
        values=f"{tech.lower()}_collabs"
    ).fillna(0)
    wide.columns = [f"{tech.lower()}_collabs_{int(c)}" for c in wide.columns]
    tech_results.append(wide.reset_index())

tech_df = reduce(
    lambda left, right: pd.merge(left, right, on="Linkedin_name", how="outer"),
    tech_results
)

firm_master_ipcei = firm_master_ipcei.merge(tech_df, on="Linkedin_name", how="left")
firm_master_ipcei = firm_master_ipcei.fillna(0)

firm_master_ipcei["hydrogen_collabs_pre"] = firm_master_ipcei.get("hydrogen_collabs_2020", 0)
firm_master_ipcei["battery_collabs_pre"]  = firm_master_ipcei.get("battery_collabs_2020",  0)

print("✓ Tech collaboration variables created")


COMPUTING TECH COLLAB COUNTS
✓ Tech collaboration variables created


#### 3. Prepare panels and save

In [11]:
# ============================================================
# SAVE extended firm master
# ============================================================

print("\nSaving datasets and merging into panels...")

# Save firm master
save_df(
    firm_master_ipcei,
    "interim",
    "combined",
    "firm_master_ipcei_corrected_extended_jan26.csv"
)

print("✓ Extended firm master saved")


Saving datasets and merging into panels...
✓ Extended firm master saved


In [12]:
# =========================
# Extend firm_month_outcomes
# =========================

print("\n" + "="*80)
print("EXTENDING FIRM_MONTH_OUTCOMES")
print("="*80)

# ── Prep ──────────────────────────────────────────────────────────────────────
firm_month_outcomes["month"] = pd.to_datetime(firm_month_outcomes["month"])

# Drop any previously computed collab columns to avoid duplicates on re-run
cols_to_drop = [c for c in firm_month_outcomes.columns
                if c.startswith("n_collabs_") and c != "n_collabs"]
if cols_to_drop:
    firm_month_outcomes = firm_month_outcomes.drop(columns=cols_to_drop)
    print(f"Dropped existing columns: {cols_to_drop}")

# Ensure month column on p is present (may already exist from Geographic chunk)
if "month" not in p.columns:
    p["month"] = p["post_date"].dt.to_period("M").dt.to_timestamp()

# ── Stack endpoints helper ────────────────────────────────────────────────────
def _stack_and_count(df_sub, count_name):
    src = (df_sub[["month","collaboration_id","source"]]
           .rename(columns={"source":"Linkedin_name"}))
    tgt = (df_sub[["month","collaboration_id","target"]]
           .rename(columns={"target":"Linkedin_name"}))
    ft  = pd.concat([src, tgt], ignore_index=True)
    ft  = ft.dropna(subset=["Linkedin_name","month","collaboration_id"])
    return (
        ft.groupby(["Linkedin_name","month"])["collaboration_id"]
          .nunique()
          .reset_index(name=count_name)
    )

# ── 1. Tech-specific collabs ──────────────────────────────────────────────────
for tech in ["Hydrogen", "Battery"]:
    col_name = f"n_collabs_{tech.lower()}"
    fmo_tech = _stack_and_count(p[p[tech] == 1].copy(), col_name)
    firm_month_outcomes = firm_month_outcomes.merge(
        fmo_tech, on=["Linkedin_name","month"], how="left"
    )
    firm_month_outcomes[col_name] = firm_month_outcomes[col_name].fillna(0).astype(int)
    print(f"✓ {col_name}: {(firm_month_outcomes[col_name] > 0).sum():,} obs with >0")

# ── 2. EU ecosystem (any EU operation, both firms) ────────────────────────────
fmo_eu = _stack_and_count(p[p["is_eu_ecosystem"]].copy(), "n_collabs_eu")
firm_month_outcomes = firm_month_outcomes.merge(
    fmo_eu, on=["Linkedin_name","month"], how="left"
)
firm_month_outcomes["n_collabs_eu"] = (
    firm_month_outcomes["n_collabs_eu"].fillna(0).astype(int)
)
print(f"✓ n_collabs_eu: {(firm_month_outcomes['n_collabs_eu'] > 0).sum():,} obs with >0")

# ── 3. EU primary HQ (both firms' primary country is EU) ─────────────────────
fmo_eu_primary = _stack_and_count(p[p["is_eu_primary"]].copy(), "n_collabs_eu_primary")
firm_month_outcomes = firm_month_outcomes.merge(
    fmo_eu_primary, on=["Linkedin_name","month"], how="left"
)
firm_month_outcomes["n_collabs_eu_primary"] = (
    firm_month_outcomes["n_collabs_eu_primary"].fillna(0).astype(int)
)
print(f"✓ n_collabs_eu_primary: {(firm_month_outcomes['n_collabs_eu_primary'] > 0).sum():,} obs with >0")

# ── 4. EU cross-border (both primary EU, different member states) ─────────────
fmo_eu_cb = _stack_and_count(p[p["is_eu_crossborder"]].copy(), "n_collabs_eu_crossborder")
firm_month_outcomes = firm_month_outcomes.merge(
    fmo_eu_cb, on=["Linkedin_name","month"], how="left"
)
firm_month_outcomes["n_collabs_eu_crossborder"] = (
    firm_month_outcomes["n_collabs_eu_crossborder"].fillna(0).astype(int)
)
print(f"✓ n_collabs_eu_crossborder: {(firm_month_outcomes['n_collabs_eu_crossborder'] > 0).sum():,} obs with >0")

# ── 5. Non-EU (exactly one firm has EU operation) ────────────────────────────
fmo_non_eu = _stack_and_count(p[p["is_non_eu"]].copy(), "n_collabs_non_eu")
firm_month_outcomes = firm_month_outcomes.merge(
    fmo_non_eu, on=["Linkedin_name","month"], how="left"
)
firm_month_outcomes["n_collabs_non_eu"] = (
    firm_month_outcomes["n_collabs_non_eu"].fillna(0).astype(int)
)
print(f"✓ n_collabs_non_eu: {(firm_month_outcomes['n_collabs_non_eu'] > 0).sum():,} obs with >0")

# ── 6. Cross-sector (different industry_clean, both known) ───────────────────
industry_map = dict(zip(
    firm_master_ipcei["Linkedin_name"],
    firm_master_ipcei["industry_clean"]
))
p["src_industry"] = p["source"].map(industry_map)
p["tgt_industry"] = p["target"].map(industry_map)
p["is_crosssector"]  = (
    p["src_industry"].notna() &
    p["tgt_industry"].notna() &
    (p["src_industry"] != p["tgt_industry"])
)
p["is_withinsector"] = (
    p["src_industry"].notna() &
    p["tgt_industry"].notna() &
    (p["src_industry"] == p["tgt_industry"])
)

fmo_xs = _stack_and_count(p[p["is_crosssector"]].copy(),  "n_collabs_crosssector")
fmo_ws = _stack_and_count(p[p["is_withinsector"]].copy(), "n_collabs_withinsector")

firm_month_outcomes = (
    firm_month_outcomes
    .merge(fmo_xs, on=["Linkedin_name","month"], how="left")
    .merge(fmo_ws, on=["Linkedin_name","month"], how="left")
)
firm_month_outcomes["n_collabs_crosssector"]  = (
    firm_month_outcomes["n_collabs_crosssector"].fillna(0).astype(int)
)
firm_month_outcomes["n_collabs_withinsector"] = (
    firm_month_outcomes["n_collabs_withinsector"].fillna(0).astype(int)
)
print(f"✓ n_collabs_crosssector:  {(firm_month_outcomes['n_collabs_crosssector'] > 0).sum():,} obs with >0")
print(f"✓ n_collabs_withinsector: {(firm_month_outcomes['n_collabs_withinsector'] > 0).sum():,} obs with >0")

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\nfirm_month_outcomes final shape: {firm_month_outcomes.shape}")
print(f"Columns: {list(firm_month_outcomes.columns)}")

# ── Save ──────────────────────────────────────────────────────────────────────
save_df(firm_month_outcomes, "interim", "combined",
        "firm_month_outcomes_extended_jan26.csv")
print("✓ firm_month_outcomes_extended saved")


EXTENDING FIRM_MONTH_OUTCOMES
✓ n_collabs_hydrogen: 182,081 obs with >0
✓ n_collabs_battery: 188,679 obs with >0
✓ n_collabs_eu: 369,542 obs with >0
✓ n_collabs_eu_primary: 338,137 obs with >0
✓ n_collabs_eu_crossborder: 166,465 obs with >0
✓ n_collabs_non_eu: 502,680 obs with >0
✓ n_collabs_crosssector:  705,341 obs with >0
✓ n_collabs_withinsector: 465,283 obs with >0

firm_month_outcomes final shape: (986263, 11)
Columns: ['Linkedin_name', 'month', 'n_collabs', 'n_collabs_hydrogen', 'n_collabs_battery', 'n_collabs_eu', 'n_collabs_eu_primary', 'n_collabs_eu_crossborder', 'n_collabs_non_eu', 'n_collabs_crosssector', 'n_collabs_withinsector']
✓ firm_month_outcomes_extended saved
